# 03 - Transformation (KDD stage 3)
* antecedent precipitation index (API, decay 0.85) from cleaned rainfall
* 14-day lookback windows per region (LSTM input)
* satellite tile at the window's last day (CNN input)
* **chronological** split by date (no random shuffling: avoids temporal leakage)
* scalers fitted on the training split only
* target: flood event on the day AFTER the window ends

In [ ]:
import json
import sys
from pathlib import Path

try:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks" / "common.py").exists())
except StopIteration:
    raise SystemExit("notebooks/common.py not found. Run inside the repo.")
sys.path.insert(0, str(ROOT / "notebooks"))

import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view
from common import set_seed, sha256_file, write_meta

set_seed()
df = (pd.read_csv(ROOT / "data" / "ewsa_clean.csv", parse_dates=["date"])
        .sort_values(["region_id", "date"]).reset_index(drop=True))
tiles_all = np.load(ROOT / "data" / "ewsa_tiles.npz")["tiles"]


def api_series(rain: np.ndarray, k: float = 0.85) -> np.ndarray:
    out = np.zeros_like(rain, dtype=float)
    out[0] = rain[0]
    for t in range(1, len(rain)):
        out[t] = k * out[t - 1] + rain[t]
    return out


df["api"] = df.groupby("region_id")["rainfall_mm"].transform(
    lambda s: pd.Series(api_series(s.to_numpy()), index=s.index))

In [ ]:
L = 14
FEATS = ["rainfall_mm", "river_level_m", "temperature_c", "humidity_pct", "pressure_hpa",
         "wind_ms", "soil_moisture", "api", "river_level_m_missing", "soil_moisture_missing",
         "susceptibility_idx", "impervious_frac"]
seqs, tls, ys, dts, regs = [], [], [], [], []
for rid, g in df.groupby("region_id", sort=True):
    g = g.reset_index(drop=True)
    n = len(g)
    arr = g[FEATS].to_numpy(dtype=np.float32)
    win = sliding_window_view(arr, L, axis=0).transpose(0, 2, 1)[: n - L]  # (n-L, L, F)
    end = np.arange(L - 1, n - 1)
    seqs.append(win)
    ys.append(g["event"].to_numpy()[end + 1])
    dts.append(g["date"].to_numpy()[end])
    regs.append(np.full(len(end), rid))
    tls.append(tiles_all[g["row_id"].to_numpy()[end]])
X, T = np.concatenate(seqs), np.concatenate(tls)
y, dates, region = np.concatenate(ys).astype(np.float32), np.concatenate(dts), np.concatenate(regs)
print(X.shape, T.shape, y.shape)

In [ ]:
tr = dates < np.datetime64("2021-07-01")
va = (dates >= np.datetime64("2021-07-01")) & (dates < np.datetime64("2022-04-01"))
te = dates >= np.datetime64("2022-04-01")
for name, m in (("train", tr), ("val", va), ("test", te)):
    print(f"{name:5s} n={m.sum():6d} positives={int(y[m].sum()):4d} prevalence={y[m].mean():.3f}")
assert y[te].sum() >= 20, "too few test positives for stable metrics"

F = X.shape[2]
mean = X[tr].reshape(-1, F).mean(0)
std = X[tr].reshape(-1, F).std(0) + 1e-6
Xs = ((X - mean) / std).astype(np.float32)
tmean = T[tr].mean(axis=(0, 2, 3), keepdims=True)[0]
tstd = T[tr].std(axis=(0, 2, 3), keepdims=True)[0] + 1e-6
Ts = ((T - tmean) / tstd).astype(np.float32)

rain_i = FEATS.index("rainfall_mm")
tab = np.concatenate([Xs[:, -1, :], Xs[:, -3:, rain_i].sum(1, keepdims=True),
                      Xs[:, -7:, rain_i].sum(1, keepdims=True)], axis=1).astype(np.float32)

In [ ]:
arrays = {}
for name, m in (("train", tr), ("val", va), ("test", te)):
    arrays[f"seq_{name}"], arrays[f"tile_{name}"] = Xs[m], Ts[m]
    arrays[f"tab_{name}"], arrays[f"y_{name}"] = tab[m], y[m]
    arrays[f"date_{name}"] = dates[m].astype("datetime64[D]").astype(str)
    arrays[f"region_{name}"] = region[m]
out = ROOT / "data" / "ewsa_dataset_v1.npz"
np.savez_compressed(out, **arrays)

(ROOT / "models").mkdir(exist_ok=True)
prep = {"lookback": L, "features": FEATS, "seq_mean": mean.tolist(), "seq_std": std.tolist(),
        "tile_mean": tmean.ravel().tolist(), "tile_std": tstd.ravel().tolist(),
        "api_decay": 0.85, "splits": {"train": "< 2021-07-01", "val": "2021-07-01..2022-03-31",
                                      "test": ">= 2022-04-01"}}
(ROOT / "models" / "preprocessing.json").write_text(json.dumps(prep, indent=2), encoding="utf-8")
write_meta(ROOT / "data" / "ewsa_dataset_v1.meta.json", data_kind="synthetic",
           sha256={out.name: sha256_file(out), "ewsa_clean.csv": sha256_file(ROOT / "data" / "ewsa_clean.csv")})
print("saved", out.name)

In [ ]:
{
 "cells": [
  {"cell_type": "markdown", "metadata": {}, "source": "# KDD Stage 3 — Transformation\n\nBuilds the two model inputs: `X_seq` (LSTM branch: rolling time-series features including\nthe antecedent precipitation index and land-cover composition) and `X_tile` (CNN branch:\nsatellite tiles), aligned per (region, day), with a fixed lookback window."},
  {"cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": "import numpy as np\nimport pandas as pd\nimport hashlib, json\nfrom pathlib import Path\nfrom sklearn.model_selection import train_test_split\n\nSEED = 42\nLOOKBACK = 10\nPROC = Path(\"../data/processed\")\nRAW = Path(\"../data/raw\")\n\nstations = pd.read_parquet(PROC / \"station_readings_clean.parquet\")\nland_cover = pd.read_parquet(PROC / \"land_cover_clean.parquet\").set_index(\"region_id\")\ntiles_npz = np.load(RAW / \"satellite_tiles.npz\")\n\nFEATURES = [\"rainfall_mm_24h\", \"river_level_m\", \"soil_moisture\", \"temperature_c\", \"humidity_pct\"]\nMISSING_FLAGS = [f\"{c}_missing\" for c in FEATURES]\nLC = [\"urban_pct\", \"water_pct\", \"vegetation_pct\", \"bare_pct\"]"},
  {"cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": "def api_index(rainfall: np.ndarray, decay: float = 0.9) -> np.ndarray:\n    api = np.zeros_like(rainfall)\n    for i in range(1, len(rainfall)):\n        api[i] = decay * api[i - 1] + rainfall[i]\n    return api\n\nsequences, tile_inputs, labels, meta = [], [], [], []\nfor region_id, df in stations.groupby(\"region_id\"):\n    df = df.sort_values(\"date\").reset_index(drop=True)\n    df[\"api_index\"] = api_index(df[\"rainfall_mm_24h\"].values)\n    lc_row = land_cover.loc[region_id, LC].values.astype(\"float32\")\n    tiles = tiles_npz[region_id]\n    feat_cols = FEATURES + [\"api_index\"] + MISSING_FLAGS\n    values = df[feat_cols].values.astype(\"float32\")\n\n    for i in range(LOOKBACK, len(df)):\n        window = values[i - LOOKBACK:i]\n        lc_tiled = np.tile(lc_row, (LOOKBACK, 1))\n        sequences.append(np.concatenate([window, lc_tiled], axis=1))\n        tile_inputs.append(tiles[i])\n        labels.append(int(df[\"flood_label\"].iloc[i]))\n        meta.append((region_id, str(df[\"date\"].iloc[i].date())))\n\nX_seq = np.stack(sequences).astype(\"float32\")\nX_tile = np.stack(tile_inputs).astype(\"float32\")\ny = np.array(labels, dtype=\"int64\")\nprint(X_seq.shape, X_tile.shape, y.shape, \"positive rate:\", y.mean().round(3))"},
  {"cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": "# Normalise sequence features using TRAIN split statistics only (no leakage), split is\n# chronological-ish via stratified random split for a prototype; flag this choice in\n# 05_interpretation.ipynb rather than hide it.\nidx = np.arange(len(y))\ntrain_idx, temp_idx = train_test_split(idx, test_size=0.3, random_state=SEED, stratify=y)\nval_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=SEED, stratify=y[temp_idx])\n\nmean = X_seq[train_idx].mean(axis=(0, 1), keepdims=True)\nstd = X_seq[train_idx].std(axis=(0, 1), keepdims=True) + 1e-6\nX_seq_norm = (X_seq - mean) / std\n\nPROC_NPZ = PROC / \"model_ready.npz\"\nnp.savez_compressed(\n    PROC_NPZ,\n    X_seq=X_seq_norm, X_tile=X_tile, y=y,\n    train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,\n    seq_mean=mean, seq_std=std,\n)\n\ndef sha256_of(path):\n    h = hashlib.sha256()\n    with open(path, \"rb\") as f:\n        for chunk in iter(lambda: f.read(8192), b\"\"):\n            h.update(chunk)\n    return h.hexdigest()\n\n(PROC / \"CHECKSUMS.sha256.json\").write_text(json.dumps({PROC_NPZ.name: sha256_of(PROC_NPZ)}, indent=2))\nprint(\"train/val/test sizes:\", len(train_idx), len(val_idx), len(test_idx))"}
 ],
 "metadata": {"kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"}, "language_info": {"name": "python", "version": "3.10"}},
 "nbformat": 4,
 "nbformat_minor": 5
}